# YOLO Foundations Part 4: Model Evaluation

In this lab, we will compare different YOLO models and check for overfitting.

## 1. Environment Setup

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Load Models

Load two different YOLO model sizes (nano and small) to compare.

In [ ]:
from ultralytics import YOLO
import time

# Load models
model_n = YOLO("yolo26n.pt")
model_s = YOLO("yolo26s.pt")

## 3. Compare Models

Run validation on both models and compare mAP and inference time.

In [ ]:
source_img = "https://ultralytics.com/images/bus.jpg"

print("Testing inference time (3 runs each):")

# Time nano model
times_n = []
for _ in range(3):
    start = time.time()
    model_n.predict(source=source_img)
    times_n.append(time.time() - start)
avg_time_n = sum(times_n) / len(times_n)

# Time small model
times_s = []
for _ in range(3):
    start = time.time()
    model_s.predict(source=source_img)
    times_s.append(time.time() - start)
avg_time_s = sum(times_s) / len(times_s)

print(f"  yolo26n: {avg_time_n:.3f}s")
print(f"  yolo26s: {avg_time_s:.3f}s")

In [ ]:
# Run validation
metrics_n = model_n.val(data="coco8.yaml")
metrics_s = model_s.val(data="coco8.yaml")

print("\n" + "="*50)
print("Model Comparison Summary")
print("="*50)
print(f"{'Model':<12} {'mAP50':<10} {'mAP50-95':<10} {'Time (s)':<10}")
print("-"*50)
print(f"{'yolo26n':<12} {metrics_n.box.map50:<10.4f} {metrics_n.box.map:<10.4f} {avg_time_n:<10.3f}")
print(f"{'yolo26s':<12} {metrics_s.box.map50:<10.4f} {metrics_s.box.map:<10.4f} {avg_time_s:<10.3f}")
print("="*50)

### 💡 Exercise: Manual IoU Verification

Given two boxes `box1 = [100, 100, 200, 200]` and `box2 = [150, 150, 250, 250]`, approximate the **Intersection over Union (IoU)**.

In [ ]:
# TODO: Calculate Intersection and Union areas manually


<details>
<summary>Solution</summary>

```python
# Intersection: [150, 150, 200, 200] -> area = 50 * 50 = 2500
# Box1 area: 100 * 100 = 10000
# Box2 area: 100 * 100 = 10000
# Union: 10000 + 10000 - 2500 = 17500
# IoU: 2500 / 17500 = 0.142
```

</details>

## 4. Overfitting Check

Compare train mAP vs val mAP to check for overfitting.

In [ ]:
# Run train and val to get both metrics
model_test = YOLO("yolo26n.pt")
model_test.train(data="coco8.yaml", epochs=10, imgsz=640, verbose=False)

# Get train metrics (from last epoch)
val_metrics = model_test.val(data="coco8.yaml")

# Simulate train metrics (in real scenario, load from results.csv)
# For demo purposes, let's assume train mAP ~ val mAP for small dataset
train_map = val_metrics.box.map50  # This would be from training logs
val_map = val_metrics.box.map50

print("\nOverfitting Check:")
print(f"  Train mAP: {train_map:.4f}")
print(f"  Val mAP:   {val_map:.4f}")

if val_map < train_map * 0.9:
    gap_pct = (1 - val_map/train_map) * 100
    print(f"\n⚠️  Warning: Validation mAP is {gap_pct:.1f}% lower than train mAP!")
    print("   This indicates potential overfitting.")
    print("   Consider: more data, augmentation, or regularization")
else:
    print("\n✅ No overfitting detected. Train and val mAP are comparable.")

---
## End of Lab
You have compared models and checked for overfitting.
Use these techniques to pick the best model for production.